In [1]:
# pip install -U xgboost
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math
from catboost import CatBoostClassifier
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    f1_score,
    accuracy_score
)
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder
df_og = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
df2 = pd.read_csv('https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv')


In [2]:
tf = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

In [3]:
tf_id = tf['id']   # ✅ save id column first

In [4]:
# tf_id = tf.pop('id')

In [5]:
# df_og.head()

In [6]:
# df2.head()

In [7]:
id_col = ['id', 'customerID']

In [8]:
df_og.describe(include='object')

,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Churn
count,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194
unique,2,2,2,2,3,3,3,3,3,3,3,3,3,2,4,2
top,Female,Yes,No,Yes,No,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,No
freq,298738,309554,414362,557893,283384,272386,289474,250083,247377,288571,240301,241435,298918,365579,215372,460377


In [9]:
df2.describe(include='object')

,customerID,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,TotalCharges,Churn
count,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043
unique,7043,2,2,2,2,3,3,3,3,3,3,3,3,3,2,4,6531,2
top,3186-AJIEK,Male,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,,No
freq,1,3555,3641,4933,6361,3390,3096,3498,3088,3095,3473,2810,2785,3875,4171,2365,11,5174


##### no null values were observed

In [10]:
df2.shape

(7043, 21)

In [11]:
df_og.shape

(594194, 21)

In [12]:
df = pd.concat([df_og, df2], axis=0)

In [13]:
# df.head()

In [20]:
# tf.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check,115.55,8061.50
1,Female,0,Yes,No,71,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.80,1336.50
2,Male,0,No,No,12,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),55.55,633.55
3,Male,0,Yes,Yes,71,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),84.10,6457.15
4,Female,0,No,No,15,Yes,No,Fiber optic,Yes,No,No,No,Yes,Yes,Month-to-month,No,Electronic check,90.35,1233.65


In [22]:
df.drop(columns=id_col, inplace=True)
tf.drop(columns="id", inplace=True)

In [48]:
# df.shape

In [23]:
numerical_features_list= ["tenure","MonthlyCharges","TotalCharges"]

for col in df.columns:
  if col not in numerical_features_list:
    print(col, df[col].unique())
    print("*"*50)

gender ['Male' 'Female']
**************************************************
SeniorCitizen [0 1]
**************************************************
Partner ['Yes' 'No']
**************************************************
Dependents ['Yes' 'No']
**************************************************
PhoneService ['Yes' 'No']
**************************************************
MultipleLines ['No' 'Yes' 'No phone service']
**************************************************
InternetService ['DSL' 'Fiber optic' 'No']
**************************************************
OnlineSecurity ['Yes' 'No' 'No internet service']
**************************************************
OnlineBackup ['No' 'Yes' 'No internet service']
**************************************************
DeviceProtection ['Yes' 'No' 'No internet service']
**************************************************
TechSupport ['Yes' 'No' 'No internet service']
**************************************************
StreamingTV ['No' 'Yes' 'No internet 

In [24]:
# df.isnull().sum()

In [25]:
print(len(df[df["TotalCharges"]==' ']))
df[df["TotalCharges"]==' ']
# it can be observed as Tenure is 0 TotalCharges is ' ', so replace ' ' with 0
df["TotalCharges"] = df["TotalCharges"].replace({" ":"0"})
tf["TotalCharges"] = tf["TotalCharges"].replace({" ":"0"})

11


In [26]:
df["TotalCharges"]=df["TotalCharges"].astype(float)
tf["TotalCharges"]=tf["TotalCharges"].astype(float)

In [27]:
df.dtypes

gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object

In [28]:
# print(f"Churn Distribution: {df['Churn'].value_counts()}")

In [29]:
# def plot_boxplot(df,column_name):

#   plt.figure(figsize=(5,3))
#   sns.boxplot(y=df[column_name])
#   plt.title(f"Distribution of {column_name}")
#   plt.ylabel(column_name)
#   plt.show

In [30]:
# plot_boxplot(df,"tenure")


In [31]:
# plot_boxplot(df,"MonthlyCharges")


In [32]:
# plot_boxplot(df,"TotalCharges")


In [33]:
# object_cols= df.select_dtypes(include="object").columns.to_list()
# object_cols= ["SeniorCitizen"]+ object_cols
# # object_cols

# for col in object_cols:
#   plt.figure(figsize=(5,3))
#   sns.countplot(x=df[col])
#   plt.title(f"Count Plot of {col}")
#   plt.show()
#   print(" "*50)

In [35]:
df["Churn"]=df["Churn"].replace({"Yes":1,"No":0})
# tf["Churn"]=tf["Churn"].replace({"Yes":1,"No":0})


In [36]:
object_columns= df.select_dtypes(include="object").columns
     
print(object_columns)

print("-"*20)

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')
--------------------


In [37]:
import pickle

In [38]:
# inititalzie a dictonary to save the encoders
encoders = {}

# applying label encodeing and store the encoders

for column in object_columns:
  label_encoder=LabelEncoder()
  df[column]= label_encoder.fit_transform(df[column])
  encoders[column]= label_encoder

with open("encoders.pkl","wb") as f:
  pickle.dump(encoders,f)

In [39]:
# encoders

{'gender': LabelEncoder(),
 'Partner': LabelEncoder(),
 'Dependents': LabelEncoder(),
 'PhoneService': LabelEncoder(),
 'MultipleLines': LabelEncoder(),
 'InternetService': LabelEncoder(),
 'OnlineSecurity': LabelEncoder(),
 'OnlineBackup': LabelEncoder(),
 'DeviceProtection': LabelEncoder(),
 'TechSupport': LabelEncoder(),
 'StreamingTV': LabelEncoder(),
 'StreamingMovies': LabelEncoder(),
 'Contract': LabelEncoder(),
 'PaperlessBilling': LabelEncoder(),
 'PaymentMethod': LabelEncoder()}

In [40]:
# normalizing the monthlyCharges for df without bin

from sklearn.preprocessing import StandardScaler
import pickle

scaler = StandardScaler()
df['MonthlyCharges'] = scaler.fit_transform(df[['MonthlyCharges']])

# Save the scaler
with open('monthlycharges_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

df['TotalCharges'] = scaler.fit_transform(df[['TotalCharges']])

# Save the scaler
with open('totalcharges_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)


df['tenure'] = scaler.fit_transform(df[['tenure']])

# Save the scaler
with open('tenure.pkl', 'wb') as f:
    pickle.dump(scaler, f)


In [41]:
# df.head()

In [42]:
X= df.drop(columns=['Churn'])
y=df['Churn']

In [43]:
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2, random_state=42)


In [44]:
print(y_train.shape)
print(y_train.value_counts())

(480989,)
Churn
0    372553
1    108436
Name: count, dtype: int64


### Trying Downsampling


In [45]:
import pandas as pd

X = X_train.copy()
y = y_train.copy()

df = pd.concat([X, y], axis=1)

majority = df[df[y.name] == 0]
minority = df[df[y.name] == 1]

majority_downsampled = majority.sample(
    n=len(minority),
    random_state=42
)

df_balanced = pd.concat([majority_downsampled, minority])

X_train_down = df_balanced.drop(columns=[y.name])
y_train_down = df_balanced[y.name]

In [46]:
print(y_train_down.shape)
print(y_train_down.value_counts())

(216872,)
Churn
0    108436
1    108436
Name: count, dtype: int64


In [58]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import randint, uniform

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

param_dist = {
    'XGBoost': {
        'n_estimators': randint(200, 600),
        'max_depth': randint(4, 12),
        'learning_rate': uniform(0.01, 0.15),
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.6, 0.4)
    }
}

models = {
    'XGBoost': XGBClassifier(
        random_state=42,
        tree_method='hist',
        device='cuda',
        eval_metric='logloss',
        max_bin=256,
        verbosity=0
    )
}

cv_scores = {}

for name, model in models.items():
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist[name],
        n_iter=12,
        scoring='roc_auc',
        cv=skf,
        n_jobs=-1,
        random_state=42,
        verbose=1
    )

    search.fit(X_train_down, y_train_down)

    cv_scores[name] = search.best_score_

    print(name, search.best_score_, search.best_params_)

print(cv_scores)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
XGBoost 0.9142269213455535 {'colsample_bytree': np.float64(0.6923575302488596), 'learning_rate': np.float64(0.04615381990390176), 'max_depth': 7, 'n_estimators': 463, 'subsample': np.float64(0.6137554084460873)}
{'XGBoost': np.float64(0.9142269213455535)}


Fitting 3 folds for each of 12 candidates, totalling 36 fits
XGBoost 0.9142269213455535 {'colsample_bytree': np.float64(0.6923575302488596), 'learning_rate': np.float64(0.04615381990390176), 'max_depth': 7, 'n_estimators': 463, 'subsample': np.float64(0.6137554084460873)}
Fitting 3 folds for each of 12 candidates, totalling 36 fits

In [59]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import randint, uniform

from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

param_dist = {
    'CatBoost': {
        'depth': randint(5, 9),
        'learning_rate': uniform(0.02, 0.1),
        'iterations': randint(300, 700),
        'l2_leaf_reg': uniform(2, 4)
    },
    'Random Forest': {
        'n_estimators': randint(200, 400),
        'max_depth': randint(10, 30),
        'min_samples_split': randint(2, 8),
        'max_features': ['sqrt']
    }
}

models = {
    'CatBoost': CatBoostClassifier(
        task_type="GPU",
        devices='0',
        verbose=0,
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        random_state=42,
        n_jobs=1
    )
}

cv_scores = {}

for name, model in models.items():
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist[name],
        n_iter=5,
        scoring='roc_auc',
        cv=skf,
        n_jobs=1,
        random_state=42,
        verbose=1
    )

    search.fit(X_train_down, y_train_down)

    cv_scores[name] = search.best_score_

    print(name, search.best_score_, search.best_params_)

print(cv_scores)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
CatBoost 0.9142571997620559 {'depth': 6, 'iterations': 593, 'l2_leaf_reg': np.float64(2.0031150633640573), 'learning_rate': np.float64(0.11922115592912176)}
Fitting 3 folds for each of 5 candidates, totalling 15 fits
Random Forest 0.9109489317042955 {'max_depth': 16, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 292}
{'CatBoost': np.float64(0.9142571997620559), 'Random Forest': np.float64(0.9109489317042955)}


Fitting 3 folds for each of 5 candidates, totalling 15 fits
CatBoost 0.9142571997620559 {'depth': 6, 'iterations': 593, 'l2_leaf_reg': np.float64(2.0031150633640573), 'learning_rate': np.float64(0.11922115592912176)}
Fitting 3 folds for each of 5 candidates, totalling 15 fits
Random Forest 0.9109489317042955 {'max_depth': 16, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 292}
{'CatBoost': np.float64(0.9142571997620559), 'Random Forest': np.float64(0.9109489317042955)}

### Saving Models

In [47]:
import pickle

xgb_best = XGBClassifier(
    colsample_bytree=0.6923575302488596,
    learning_rate=0.04615381990390176,
    max_depth=7,
    n_estimators=463,
    subsample=0.6137554084460873,
    tree_method='hist',
    device='cuda',
    eval_metric='logloss',
    verbosity=0,
    random_state=42
)

cat_best = CatBoostClassifier(
    depth=6,
    iterations=593,
    l2_leaf_reg=2.0031150633640573,
    learning_rate=0.11922115592912176,
    task_type="CPU",
    devices='0',
    verbose=0,
    random_state=42
)

rf_best = RandomForestClassifier(
    max_depth=16,
    max_features='sqrt',
    min_samples_split=5,
    n_estimators=292,
    n_jobs=1,
    random_state=42
)

In [48]:
xgb_best.fit(X_train_down, y_train_down)
cat_best.fit(X_train_down, y_train_down)
rf_best.fit(X_train_down, y_train_down)

RandomForestClassifier(max_depth=16, min_samples_split=5, n_estimators=292,
                       n_jobs=1, random_state=42)

In [49]:
with open("xgb_model.pkl", "wb") as f:
    pickle.dump(xgb_best, f)

with open("cat_model.pkl", "wb") as f:
    pickle.dump(cat_best, f)

with open("rf_model.pkl", "wb") as f:
    pickle.dump(rf_best, f)

### working with tf

In [50]:
input_data_df=tf[X_train.columns]

In [51]:
# 1. Load encoders
with open("encoders.pkl", "rb") as f:
    encoders = pickle.load(f)

# 2. Label encode all categorical features
for column, encoder in encoders.items():
    input_data_df[column] = encoder.transform(input_data_df[column])

# 3. Load scalers
with open("monthlycharges_scaler.pkl", "rb") as f:
    monthly_scaler = pickle.load(f)

with open("totalcharges_scaler.pkl", "rb") as f:
    total_scaler = pickle.load(f)

with open("tenure.pkl","rb") as f:
    tenure_scaler= pickle.load(f)

# 4. Apply scaling
input_data_df["MonthlyCharges"] = monthly_scaler.transform(input_data_df[["MonthlyCharges"]])
input_data_df["TotalCharges"] = total_scaler.transform(input_data_df[["TotalCharges"]])
input_data_df["tenure"] = tenure_scaler.transform(input_data_df[["tenure"]])

In [59]:
prediction_xgb = xgb_best.predict(input_data_df)
pred_prob_xgb = xgb_best.predict_proba(input_data_df)

In [56]:
prediction_cat = cat_best.predict(input_data_df)
pred_prob_cat = cat_best.predict_proba(input_data_df)

In [57]:
prediction_rf = rf_best.predict(input_data_df)
pred_prob_rf = rf_best.predict_proba(input_data_df)

In [61]:
# pred_prob_xgb.shape  # (n, 2)

In [55]:
# # --- PREDICTIONS ---
# cat_preds = cat.predict(X_test)
# lgb_preds = lgbm.predict(X_test)
# dtest = xgb.DMatrix(X_test)
# xgb_preds = (model.predict(dtest) >= 0.5).astype(int)

# stack = np.vstack([cat_preds, lgb_preds, xgb_preds])

# # majority vote
# final_preds = np.round(stack.mean(axis=0)).astype(int)

# # --- PROBABILITIES ---
# cat_prob = cat.predict_proba(X_test)[:, 1]
# lgb_prob = lgbm.predict_proba(X_test)[:, 1]
# xgb_prob = model.predict(dtest)

# # weighted average (tune weights)
# avg_prob = (
#     0.30 * cat_prob +
#     0.40 * lgb_prob +
#     0.30 * xgb_prob
# )

# # --- METRICS ---
# metrics = {
#     "Accuracy": accuracy_score(y_test, final_preds),
#     "F1": f1_score(y_test, final_preds),
#     "ROC-AUC": roc_auc_score(y_test, avg_prob)
# }

# print(metrics)

In [62]:
# --- ALIGN COLUMNS ---
tf = input_data_df[X_train.columns].copy()

# --- PREDICTIONS ---
xgb_preds = prediction_xgb
cat_preds = prediction_cat
rf_preds  = prediction_rf

# stack predictions
stack = np.vstack([cat_preds, rf_preds, xgb_preds])

# majority vote
final_preds = np.round(stack.mean(axis=0)).astype(int)

# --- PROBABILITIES ---
xgb_prob = pred_prob_xgb[:, 1]
cat_prob = pred_prob_cat[:, 1]
rf_prob  = pred_prob_rf[:, 1]

# weighted average (you can tune weights)
avg_prob = (
    0.30 * cat_prob +
    0.40 * rf_prob +
    0.30 * xgb_prob
)

# --- SUBMISSION ---
submission_ensemble = pd.DataFrame({
    "id": tf_id,
    "Churn": avg_prob
})

submission_ensemble.to_csv("ensemble_predictions.csv", index=False)

print("Submission created")

Submission created
